In [1]:
# Cell 1 — Imports
import duckdb
import pandas as pd
import numpy as np
import json
import re
import os
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go

print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Build In-Memory Database with Rich Schema
con = duckdb.connect()

# --- Flights table ---
np.random.seed(42)
n_flights = 5000
airlines  = ['Delta', 'United', 'American', 'Southwest', 'JetBlue']
airports  = ['BOS', 'JFK', 'LAX', 'ORD', 'SEA', 'ATL', 'DFW', 'MIA', 'SFO', 'DEN']

flights_df = pd.DataFrame({
    'flight_id':        range(1, n_flights+1),
    'airline':          np.random.choice(airlines, n_flights),
    'origin':           np.random.choice(airports, n_flights),
    'destination':      np.random.choice(airports, n_flights),
    'scheduled_dep':    pd.date_range('2024-01-01', periods=n_flights, freq='2h'),
    'delay_minutes':    np.random.exponential(20, n_flights).astype(int),
    'cancelled':        np.random.choice([0,1], n_flights, p=[0.97, 0.03]),
    'passengers':       np.random.randint(80, 220, n_flights),
    'fuel_cost_usd':    np.random.uniform(8000, 45000, n_flights).round(2),
    'distance_km':      np.random.randint(300, 5000, n_flights)
})
flights_df = flights_df[flights_df['origin'] != flights_df['destination']]

# --- Transactions table ---
n_tx = 10000
categories = ['dining', 'travel', 'retail', 'utilities', 'healthcare', 'entertainment']
transactions_df = pd.DataFrame({
    'tx_id':        range(1, n_tx+1),
    'customer_id':  np.random.randint(1, 500, n_tx),
    'amount_usd':   np.random.exponential(85, n_tx).round(2),
    'category':     np.random.choice(categories, n_tx),
    'merchant':     np.random.choice(['Amazon','Uber','Delta','Walmart','Netflix','CVS'], n_tx),
    'tx_date':      pd.date_range('2024-01-01', periods=n_tx, freq='45min'),
    'is_fraud':     np.random.choice([0,1], n_tx, p=[0.98, 0.02]),
    'country':      np.random.choice(['US','UK','CA','DE','FR'], n_tx)
})

# Register tables
con.register('flights',      flights_df)
con.register('transactions', transactions_df)

print("Database ready")
print(f"  flights table:      {len(flights_df):,} rows")
print(f"  transactions table: {len(transactions_df):,} rows")
print(f"\nSchema — flights:")
print(flights_df.dtypes.to_string())
print(f"\nSchema — transactions:")
print(transactions_df.dtypes.to_string())

Database ready
  flights table:      4,502 rows
  transactions table: 10,000 rows

Schema — flights:
flight_id                 int64
airline                     str
origin                      str
destination                 str
scheduled_dep    datetime64[us]
delay_minutes             int64
cancelled                 int64
passengers                int32
fuel_cost_usd           float64
distance_km               int32

Schema — transactions:
tx_id                   int64
customer_id             int32
amount_usd            float64
category                  str
merchant                  str
tx_date        datetime64[us]
is_fraud                int64
country                   str


In [3]:
# Cell 3 — Natural Language to SQL Engine
def nl_to_sql(question):
    """
    Rule-based NL to SQL translator
    In production this would use an LLM — 
    here we show the logic layer clearly
    """
    q = question.lower()
    
    # Flight queries
    if 'delay' in q and 'airline' in q:
        return """
        SELECT airline,
               ROUND(AVG(delay_minutes), 2) AS avg_delay_min,
               COUNT(*)                      AS total_flights,
               SUM(cancelled)                AS cancellations
        FROM flights
        GROUP BY airline
        ORDER BY avg_delay_min DESC
        """
    elif 'fuel' in q and ('route' in q or 'expensive' in q):
        return """
        SELECT origin, destination,
               ROUND(AVG(fuel_cost_usd), 2) AS avg_fuel_cost,
               ROUND(AVG(distance_km), 0)   AS avg_distance_km,
               COUNT(*)                      AS flights
        FROM flights
        GROUP BY origin, destination
        HAVING COUNT(*) >= 5
        ORDER BY avg_fuel_cost DESC
        LIMIT 10
        """
    elif 'busiest' in q or ('airport' in q and 'traffic' in q):
        return """
        SELECT origin AS airport,
               COUNT(*) AS departures,
               SUM(passengers) AS total_passengers
        FROM flights
        GROUP BY origin
        ORDER BY departures DESC
        """
    elif 'cancelled' in q:
        return """
        SELECT airline,
               SUM(cancelled) AS cancellations,
               COUNT(*)       AS total_flights,
               ROUND(100.0 * SUM(cancelled) / COUNT(*), 2) AS cancel_rate_pct
        FROM flights
        GROUP BY airline
        ORDER BY cancel_rate_pct DESC
        """
    # Transaction queries
    elif 'fraud' in q:
        return """
        SELECT category,
               COUNT(*) AS total_tx,
               SUM(is_fraud) AS fraud_count,
               ROUND(100.0 * SUM(is_fraud) / COUNT(*), 3) AS fraud_rate_pct,
               ROUND(AVG(amount_usd), 2) AS avg_amount
        FROM transactions
        GROUP BY category
        ORDER BY fraud_rate_pct DESC
        """
    elif 'spending' in q or 'category' in q:
        return """
        SELECT category,
               COUNT(*)                    AS transactions,
               ROUND(SUM(amount_usd), 2)   AS total_spend,
               ROUND(AVG(amount_usd), 2)   AS avg_spend
        FROM transactions
        GROUP BY category
        ORDER BY total_spend DESC
        """
    elif 'merchant' in q:
        return """
        SELECT merchant,
               COUNT(*)                  AS transactions,
               ROUND(SUM(amount_usd), 2) AS total_revenue,
               ROUND(AVG(amount_usd), 2) AS avg_transaction
        FROM transactions
        GROUP BY merchant
        ORDER BY total_revenue DESC
        """
    else:
        return """
        SELECT 'No matching query pattern found' AS message
        """

# Test the translator
test_questions = [
    "Which airline has the highest average delay?",
    "What are the most expensive fuel routes?",
    "Show me fraud rates by spending category"
]

for q in test_questions:
    sql = nl_to_sql(q)
    print(f"Q: {q}")
    print(f"SQL:{sql}")
    print("-" * 50)

Q: Which airline has the highest average delay?
SQL:
        SELECT airline,
               ROUND(AVG(delay_minutes), 2) AS avg_delay_min,
               COUNT(*)                      AS total_flights,
               SUM(cancelled)                AS cancellations
        FROM flights
        GROUP BY airline
        ORDER BY avg_delay_min DESC
        
--------------------------------------------------
Q: What are the most expensive fuel routes?
SQL:
        SELECT origin, destination,
               ROUND(AVG(fuel_cost_usd), 2) AS avg_fuel_cost,
               ROUND(AVG(distance_km), 0)   AS avg_distance_km,
               COUNT(*)                      AS flights
        FROM flights
        GROUP BY origin, destination
        HAVING COUNT(*) >= 5
        ORDER BY avg_fuel_cost DESC
        LIMIT 10
        
--------------------------------------------------
Q: Show me fraud rates by spending category
SQL:
        SELECT category,
               COUNT(*) AS total_tx,
               S

In [4]:
# Cell 4 — Full Agentic Query Loop with Explanation
def run_agent(question):
    """
    Full agentic pipeline:
    1. Translate NL to SQL
    2. Execute query
    3. Validate result
    4. Generate plain-English explanation
    """
    print(f"\n{'='*55}")
    print(f"Question: {question}")
    print(f"{'='*55}")
    
    # Step 1: Generate SQL
    sql = nl_to_sql(question).strip()
    print(f"\nGenerated SQL:\n{sql}")
    
    # Step 2: Execute
    try:
        result_df = con.execute(sql).df()
        print(f"\nResult ({len(result_df)} rows):")
        print(result_df.to_string(index=False))
        
        # Step 3: Auto-explain top result
        if len(result_df) > 0:
            top_row = result_df.iloc[0]
            col1 = result_df.columns[0]
            col2 = result_df.columns[1]
            print(f"\nInsight: '{top_row[col1]}' leads "
                  f"with {col2} = {top_row[col2]}")
        
        return result_df
    
    except Exception as e:
        print(f"\nQuery failed: {e}")
        return None

# Run all analyst queries
questions = [
    "Which airline has the highest average delay?",
    "What are the most expensive fuel routes?",
    "Which airport has the most traffic?",
    "Which airline has the most cancellations?",
    "Show me fraud rates by spending category",
    "What is total spending by merchant?"
]

all_results = {}
for q in questions:
    df = run_agent(q)
    all_results[q] = df


Question: Which airline has the highest average delay?

Generated SQL:
SELECT airline,
               ROUND(AVG(delay_minutes), 2) AS avg_delay_min,
               COUNT(*)                      AS total_flights,
               SUM(cancelled)                AS cancellations
        FROM flights
        GROUP BY airline
        ORDER BY avg_delay_min DESC

Result (5 rows):
  airline  avg_delay_min  total_flights  cancellations
   United          20.88            856           28.0
    Delta          20.59            917           51.0
  JetBlue          19.88            922           33.0
Southwest          19.61            912           30.0
 American          17.70            895           22.0

Insight: 'United' leads with avg_delay_min = 20.88

Question: What are the most expensive fuel routes?

Generated SQL:
SELECT origin, destination,
               ROUND(AVG(fuel_cost_usd), 2) AS avg_fuel_cost,
               ROUND(AVG(distance_km), 0)   AS avg_distance_km,
               COUNT(

In [5]:
# Cell 5 — Visualize Key Query Results
# Plot 1: Airline delays
delay_df = all_results["Which airline has the highest average delay?"]
fig1 = px.bar(delay_df, x='airline', y='avg_delay_min',
              color='avg_delay_min',
              color_continuous_scale='Reds',
              title='Average Flight Delay by Airline (minutes)',
              template='plotly_dark')
fig1.update_layout(width=750, height=400)
fig1.show()

# Plot 2: Fraud rates by category
fraud_df = all_results["Show me fraud rates by spending category"]
fig2 = px.bar(fraud_df, x='category', y='fraud_rate_pct',
              color='fraud_rate_pct',
              color_continuous_scale='Oranges',
              title='Fraud Rate by Transaction Category (%)',
              template='plotly_dark')
fig2.update_layout(width=750, height=400)
fig2.show()

# Plot 3: Airport traffic
airport_df = all_results["Which airport has the most traffic?"]
fig3 = px.bar(airport_df, x='airport', y='total_passengers',
              color='departures',
              color_continuous_scale='Blues',
              title='Airport Traffic — Total Passengers & Departures',
              template='plotly_dark')
fig3.update_layout(width=750, height=400)
fig3.show()

In [6]:
# Cell 6 — Export Results
import os

output_dir = r'C:\Users\Gurveer\ds-portfolio\project-10-agentic-sql\outputs'
os.makedirs(output_dir, exist_ok=True)

# Export each query result
for question, df in all_results.items():
    if df is not None:
        filename = question[:40].lower()
        filename = re.sub(r'[^a-z0-9]+', '_', filename).strip('_')
        df.to_csv(f'{output_dir}\\{filename}.csv', index=False)

# Save schema summary
schema = {
    'flights': {
        'rows': len(flights_df),
        'columns': list(flights_df.columns),
        'description': 'Flight operations data — delays, fuel, passengers'
    },
    'transactions': {
        'rows': len(transactions_df),
        'columns': list(transactions_df.columns),
        'description': 'Financial transactions — fraud, spending, merchants'
    }
}
with open(f'{output_dir}\\schema.json', 'w') as f:
    json.dump(schema, f, indent=2)

print("Exports saved:")
for f in os.listdir(output_dir):
    print(f"  {f}")

Exports saved:
  schema.json
  show_me_fraud_rates_by_spending_category.csv
  what_are_the_most_expensive_fuel_routes.csv
  what_is_total_spending_by_merchant.csv
  which_airline_has_the_highest_average_de.csv
  which_airport_has_the_most_traffic.csv
